In [6]:
import numpy as np
import csv
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import preprocessing
from tensorflow.keras.layers import Normalization, Resizing
from tensorflow.keras import layers
from tensorflow.keras import models
from sklearn.utils.class_weight import compute_class_weight

import pandas as pd

import seaborn as sns

from scipy import signal
from scipy.signal import butter, filtfilt, iirnotch, periodogram

In [3]:
import os
cwd = os.getcwd()
data_path = str(os.path.join(cwd, 'data'))
print(data_path)

c:\Users\aryan\Documents\GitHub\ecz-ware\training\data


In [32]:
SAMPLING_RATE = 1000
WIN_SIZE = 200
TIMESTEPS = 80
OVERLAP = 100

In [19]:
def normalize(data):
    return (data - np.mean(data)) / (np.std(data) + 1e-8)

def bandpass_filter(data, lowcut=5.0, highcut=200.0, fs=1000.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, data)

def find_noise_frequency(data, fs=1000.0):
    f, Pxx = periodogram(data, fs=fs)
    idx = np.argmax(Pxx)
    return f[idx]

def adaptive_notch_filter(data, fs=1000.0, quality=30):
    freq = find_noise_frequency(data, fs=fs)
    nyq = 0.5 * fs
    b, a = iirnotch(freq/nyq, quality)
    return filtfilt(b, a, data)

def cleanup(data):
    data = normalize(data)
    data = bandpass_filter(data)
    data = adaptive_notch_filter(data)
    return data

In [33]:
def get_sample(data, start, win_size=200):
    end = start + win_size
    return data[start:end]

def get_data(filepath):
    df = pd.read_csv(filepath)
    return df[['v1', 'v2', 'label']].to_numpy()

def clean_data(data: np.ndarray):
    v1 = data[:, 0]
    v2 = data[:, 1]
    labels = data[:, 2]

    v1_cleaned = cleanup(v1)
    v2_cleaned = cleanup(v2)

    return np.column_stack((v1_cleaned, v2_cleaned, labels))

def split_data(data: np.ndarray, split_ratio=0.8):
    num_samples = data.shape[0]
    split_index = int(num_samples * split_ratio)
    
    train_data = data[:split_index]
    test_data = data[split_index:]
    
    return train_data, test_data

def zc(x, threshold=0.01):
    """Zero Crossing Rate with threshold to reduce noise influence"""
    return np.sum(((x[:-1] * x[1:]) < 0) & (np.abs(x[:-1] - x[1:]) > threshold))

def ssc(x, threshold=0.01):
    """Slope Sign Changes with threshold"""
    diff1 = np.diff(x)
    return np.sum(((diff1[:-1] * diff1[1:]) < 0) &
                    (np.abs(diff1[:-1] - diff1[1:]) > threshold))


def get_features(data: np.ndarray): # please feed in cleaned data
    ch1 = data[:, 0]
    ch2 = data[:, 1]
    
    labels = data[:, 2]

    all_labels = []
    all_features = []

    win_len = WIN_SIZE // TIMESTEPS

    for i in range(0, len(ch1) - WIN_SIZE, OVERLAP):

        for j in range(TIMESTEPS):
            s = i + j * win_len
            
            sample_1 = get_sample(ch1, s, win_len)
            sample_2 = get_sample(ch2, s, win_len)

            mean_1 = np.mean(sample_1)
            mean_2 = np.mean(sample_2)

            std_1 = np.std(sample_1)
            std_2 = np.std(sample_2)

            wl_1 = np.sum(np.abs(np.diff(sample_1)))
            wl_2 = np.sum(np.abs(np.diff(sample_2)))

            zc_1 = zc(sample_1)
            zc_2 = zc(sample_2)

            ssc_1 = ssc(sample_1)
            ssc_2 = ssc(sample_2)

            feat1 = [mean_1, std_1, wl_1, zc_1, ssc_1]
            feat2 = [mean_2, std_2, wl_2, zc_2, ssc_2]

            feats = feat1 + feat2
            all_features.append(feats)
            all_labels.append(tf.cast(labels[i], tf.int32))

    return tf.stack(all_features, axis=0), tf.stack(all_labels, axis=0)


In [ ]:
scratch_data = np.vstack([
    get_data(os.path.join(data_path+'\\training_set_15-07-2025\\emg_scratching_raina.csv')),
    get_data(os.path.join(data_path+'\\training_set_15-07-2025\\emg_scratching_ariel.csv')),])

rest_data = np.vstack([
    get_data(os.path.join(data_path+'\\training_set_15-07-2025\\emg_rest_raina.csv')),
    get_data(os.path.join(data_path+'\\training_set_15-07-2025\\emg_rest_ariel.csv')),])

other_data = np.vstack([
    get_data(os.path.join(data_path+'\\training_set_15-07-2025\\emg_other_raina.csv')),
    get_data(os.path.join(data_path+'\\training_set_15-07-2025\\emg_other_ariel.csv')),])

scratch_data = clean_data(scratch_data)
rest_data = clean_data(rest_data)
other_data = clean_data(other_data)

print(scratch_data)

df_scratch_train, df_scratch_test = split_data(scratch_data)
df_rest_train, df_rest_test = split_data(rest_data)
df_other_train, df_other_test = split_data(other_data)

feats, labels = get_features(df_scratch_train)
print(feats.shape, labels.shape)

[[ 1.19786796 -0.11554253  1.        ]
 [ 1.21013475 -0.22147573  1.        ]
 [ 1.46802926 -0.29568539  1.        ]
 ...
 [ 0.09662728 -1.89348555  1.        ]
 [ 0.06212716 -1.50062923  1.        ]
 [ 0.00309919 -0.29978863  1.        ]]
